In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# -- Setup Paths ----------------------------------------
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'mimic-iii-clinical-database-demo-1.4'

print("[INFO] Loading MIMIC-III Demo dataset...")
print(f"Data path: {DATA_PATH}")

# -- Load Core Tables ----------------------------------
admissions   = pd.read_csv(DATA_PATH / 'ADMISSIONS.csv')
patients     = pd.read_csv(DATA_PATH / 'PATIENTS.csv')
icustays     = pd.read_csv(DATA_PATH / 'ICUSTAYS.csv')
diagnoses    = pd.read_csv(DATA_PATH / 'DIAGNOSES_ICD.csv')
labevents    = pd.read_csv(DATA_PATH / 'LABEVENTS.csv')
d_labitems   = pd.read_csv(DATA_PATH / 'D_LABITEMS.csv')

print("\n[OK] Tables loaded!")
print(f"Admissions:  {admissions.shape}")
print(f"Patients:    {patients.shape}")
print(f"ICU Stays:   {icustays.shape}")
print(f"Diagnoses:   {diagnoses.shape}")
print(f"Lab Events:  {labevents.shape}")

# -- Basic EDA ------------------------------------------
print("\n[INFO] Admissions columns:")
print(admissions.columns.tolist())

print("\n[INFO] Sample admission types:")
print(admissions['ADMISSION_TYPE'].value_counts())

print("\n[INFO] Sample discharge locations:")
print(admissions['DISCHARGE_LOCATION'].value_counts().head(8))

In [ ]:
# -- Fix: columns are lowercase ------------------------
print("[INFO] Sample admission types:")
print(admissions['admission_type'].value_counts())

print("\n[INFO] Sample discharge locations:")
print(admissions['discharge_location'].value_counts().head(8))

print("\n[INFO] Insurance types:")
print(admissions['insurance'].value_counts())

print("\n[INFO] Ethnicity:")
print(admissions['ethnicity'].value_counts().head(8))

print("\n[INFO] Date range:")
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])
print(f"Admissions from: {admissions['admittime'].min().date()} to {admissions['admittime'].max().date()}")

print("\n[INFO] Patients columns:")
print(patients.columns.tolist())
print(patients.head(3))


In [ ]:
# -- Build 30-day readmission target -------------------
print("[INFO] Building the 30-day readmission target...")

admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])

admissions = admissions.sort_values(['subject_id', 'admittime'])
admissions['next_admittime'] = admissions.groupby('subject_id')['admittime'].shift(-1)
admissions['days_to_readmission'] = (
    admissions['next_admittime'] - admissions['dischtime']
).dt.days

# Patients discharged as deceased are excluded from the readmission label.
admissions['readmitted_30d'] = (
    admissions['days_to_readmission'].between(0, 30)
    & (admissions['discharge_location'] != 'DEAD/EXPIRED')
).astype(int)

admissions['los_days'] = (
    admissions['dischtime'] - admissions['admittime']
).dt.total_seconds() / (24 * 3600)

print(f"Total admissions: {len(admissions)}")
print(f"Readmitted within 30 days: {admissions['readmitted_30d'].sum()}")
print(f"Readmission rate: {admissions['readmitted_30d'].mean():.1%}")
print("\nDischarge location among readmitted patients:")
print(admissions.loc[admissions['readmitted_30d'] == 1, 'discharge_location'].value_counts())

# -- Merge patient demographics ------------------------
patients['dob'] = pd.to_datetime(patients['dob'])
patients['dod'] = pd.to_datetime(patients['dod'])

df = admissions.merge(
    patients[['subject_id', 'gender', 'dob', 'dod', 'expire_flag']],
    on='subject_id',
    how='left',
)

df['age'] = (df['admittime'] - df['dob']).dt.days / 365.25
# MIMIC masks ages above 89, so values are capped for consistency.
df['age'] = df['age'].clip(upper=89)
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 45, 60, 75, 89],
    labels=['<45', '45-60', '60-75', '>75'],
)

print("\n[OK] Demographics merged successfully.")
print("\nAge distribution:")
print(df['age_group'].value_counts().sort_index())
print("\nGender distribution:")
print(df['gender'].value_counts())
print("\nReadmission rate by age group:")
print(df.groupby('age_group', observed=True)['readmitted_30d'].mean().round(3))


In [ ]:
# -- Refine age calculation for MIMIC date shifts ------
# MIMIC applies date offsets by patient, so ages must be capped safely.

def safe_age(admittime, dob):
    if pd.isna(admittime) or pd.isna(dob):
        return np.nan

    days = (admittime - dob).days
    age = days / 365.25
    if age > 200:
        return 89
    return min(age, 89)


df['age'] = df.apply(lambda row: safe_age(row['admittime'], row['dob']), axis=1)
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 45, 60, 75, 89],
    labels=['<45', '45-60', '60-75', '>75'],
)

print("[OK] Age calculation updated for MIMIC date shifts.")
print("\nAge summary statistics:")
print(df['age'].describe().round(1))
print("\nAge distribution:")
print(df['age_group'].value_counts().sort_index())
print("\nGender distribution:")
print(df['gender'].value_counts())
print("\nReadmission rate by age group:")
print(df.groupby('age_group', observed=True)['readmitted_30d'].mean().round(3))
print("\nReadmission rate by gender:")
print(df.groupby('gender')['readmitted_30d'].mean().round(3))
print("\nReadmission rate by insurance:")
print(df.groupby('insurance')['readmitted_30d'].mean().round(3))


In [ ]:
# -- ICD-9 Comorbidity Feature Engineering ------------
print("[INFO] Engineering comorbidity features from ICD-9 codes...")

# Lowercase columns
diagnoses.columns = diagnoses.columns.str.lower()

# Key comorbidity groups based on ICD-9 codes
# These are clinically validated groupings
def has_condition(icd_codes, prefixes):
    return any(str(code).startswith(tuple(prefixes)) 
               for code in icd_codes)

# Group diagnoses by admission
diag_grouped = diagnoses.groupby('hadm_id')['icd9_code'].apply(list).reset_index()

# Engineer binary comorbidity flags
diag_grouped['has_cardiac']      = diag_grouped['icd9_code'].apply(
    lambda x: has_condition(x, ['410','411','412','413','414','428']))

diag_grouped['has_respiratory']  = diag_grouped['icd9_code'].apply(
    lambda x: has_condition(x, ['490','491','492','493','494','496','518','519']))

diag_grouped['has_diabetes']     = diag_grouped['icd9_code'].apply(
    lambda x: has_condition(x, ['250']))

diag_grouped['has_renal']        = diag_grouped['icd9_code'].apply(
    lambda x: has_condition(x, ['580','581','582','583','584','585','586']))

diag_grouped['has_sepsis']       = diag_grouped['icd9_code'].apply(
    lambda x: has_condition(x, ['038','995','785']))

diag_grouped['has_hypertension'] = diag_grouped['icd9_code'].apply(
    lambda x: has_condition(x, ['401','402','403','404','405']))

diag_grouped['has_cancer']       = diag_grouped['icd9_code'].apply(
    lambda x: has_condition(x, ['140','141','142','143','144','145',
                                 '146','147','148','149','150','151',
                                 '152','153','154','155','156','157',
                                 '158','159','160','161','162','163',
                                 '164','165','170','171','172','173',
                                 '174','175','176','179','180','181',
                                 '182','183','184','185','186','187',
                                 '188','189','190','191','192','193',
                                 '194','195','196','197','198','199']))

# Charlson Comorbidity Index proxy (count of conditions)
comorbidity_cols = ['has_cardiac','has_respiratory','has_diabetes',
                    'has_renal','has_sepsis','has_hypertension','has_cancer']

diag_grouped['comorbidity_count'] = diag_grouped[comorbidity_cols].sum(axis=1)
diag_grouped['num_diagnoses']     = diag_grouped['icd9_code'].apply(len)

print("[OK] Comorbidity features engineered!")
print(f"\nComorbidity prevalence:")
for col in comorbidity_cols:
    pct = diag_grouped[col].mean()
    print(f"  {col:<25} {pct:.1%}")

print(f"\nAvg diagnoses per admission: {diag_grouped['num_diagnoses'].mean():.1f}")
print(f"Avg comorbidities per admission: {diag_grouped['comorbidity_count'].mean():.1f}")

# Merge into main df
df = df.merge(diag_grouped.drop('icd9_code', axis=1), 
              on='hadm_id', how='left')

print(f"\n[OK] Merged! Dataset shape: {df.shape}")
print(f"Readmission rate by comorbidity count:")
print(df.groupby('comorbidity_count')['readmitted_30d'].mean().round(3))

In [ ]:
# -- Extract Key Lab Values ----------------------------
print("[INFO] Extracting clinically important lab values...")

# Lowercase
labevents.columns = labevents.columns.str.lower()
d_labitems.columns = d_labitems.columns.str.lower()

# Key lab tests and their LOINC item IDs in MIMIC
# These are the most clinically important markers
key_labs = {
    'creatinine'    : [50912],  # kidney function
    'glucose'       : [50931],  # blood sugar
    'hemoglobin'    : [51222],  # anemia marker
    'white_blood_cell': [51301],# infection marker
    'sodium'        : [50983],  # electrolyte
    'potassium'     : [50971],  # electrolyte
    'bicarbonate'   : [50882],  # acid-base balance
    'bun'           : [51006],  # blood urea nitrogen - kidney
    'platelet'      : [51265],  # clotting
}

# Filter to key labs only
all_item_ids = [id for ids in key_labs.values() for id in ids]
labs_filtered = labevents[labevents['itemid'].isin(all_item_ids)].copy()

# Map item ID to lab name
item_to_name = {}
for name, ids in key_labs.items():
    for id in ids:
        item_to_name[id] = name

labs_filtered['lab_name'] = labs_filtered['itemid'].map(item_to_name)

# Convert value to numeric
labs_filtered['valuenum'] = pd.to_numeric(labs_filtered['valuenum'], errors='coerce')
labs_filtered = labs_filtered.dropna(subset=['valuenum', 'hadm_id'])

# Aggregate per admission - mean, min, max for each lab
lab_agg = labs_filtered.groupby(['hadm_id', 'lab_name'])['valuenum'].agg(
    ['mean', 'min', 'max', 'std', 'count']
).round(3)

lab_agg.columns = ['mean', 'min', 'max', 'std', 'count']
lab_agg = lab_agg.reset_index()

# Pivot to wide format
lab_wide = lab_agg.pivot_table(
    index='hadm_id',
    columns='lab_name',
    values=['mean', 'min', 'max'],
    aggfunc='first'
).reset_index()

# Flatten column names
lab_wide.columns = ['hadm_id'] + [
    f'{lab}_{stat}' for stat, lab in lab_wide.columns[1:]
]

print(f"[OK] Lab features extracted!")
print(f"Lab feature matrix shape: {lab_wide.shape}")
print(f"\nLab columns created:")
for col in lab_wide.columns[1:]:
    print(f"  {col}")

# Merge into main df
df = df.merge(lab_wide, on='hadm_id', how='left')
print(f"\n[OK] Final dataset shape: {df.shape}")
print(f"Missing lab values: {df[lab_wide.columns[1:]].isnull().mean().mean():.1%}")

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import xgboost as xgb
import shap
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

# -- Build the modeling dataset ------------------------
print("[INFO] Building the final feature matrix...")

DEMOGRAPHIC_FEATURES = [
    'age', 'los_days'
]

COMORBIDITY_FEATURES = [
    'has_cardiac', 'has_respiratory', 'has_diabetes',
    'has_renal', 'has_sepsis', 'has_hypertension',
    'has_cancer', 'comorbidity_count', 'num_diagnoses'
]

LAB_PREFIXES = [
    'creatinine_', 'glucose_', 'hemoglobin_',
    'white_blood_cell_', 'sodium_', 'potassium_',
    'bicarbonate_', 'bun_', 'platelet_'
]
LAB_FEATURES = [
    col for col in df.columns if any(col.startswith(prefix) for prefix in LAB_PREFIXES)
]

label_encoders = {
    'insurance': LabelEncoder(),
    'ethnicity': LabelEncoder(),
    'admission_type': LabelEncoder(),
}

df['gender_enc'] = (df['gender'] == 'M').astype(int)
for column, encoder in label_encoders.items():
    df[f'{column}_enc'] = encoder.fit_transform(df[column].fillna('Unknown'))

CATEGORICAL_FEATURES = [
    'gender_enc', 'insurance_enc', 'ethnicity_enc', 'admission_type_enc'
]
ALL_FEATURES = DEMOGRAPHIC_FEATURES + COMORBIDITY_FEATURES + LAB_FEATURES + CATEGORICAL_FEATURES

X = df[ALL_FEATURES].copy()
y = df['readmitted_30d'].copy()

mask = y.notna()
X = X.loc[mask]
y = y.loc[mask]

imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=ALL_FEATURES)

print(f"[OK] Feature matrix shape: {X_imputed.shape}")
print(f"Readmission rate: {y.mean():.1%}")
print(f"Total features: {len(ALL_FEATURES)}")
print(f"  Demographic features: {len(DEMOGRAPHIC_FEATURES)}")
print(f"  Comorbidity features: {len(COMORBIDITY_FEATURES)}")
print(f"  Laboratory features:  {len(LAB_FEATURES)}")
print(f"  Categorical features: {len(CATEGORICAL_FEATURES)}")

# -- Baseline train/test evaluation --------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"\nTraining set: {X_train.shape} | Test set: {X_test.shape}")
print(f"Training readmission rate: {y_train.mean():.1%}")
print(f"Test readmission rate:     {y_test.mean():.1%}")

mlflow.set_tracking_uri(f"file://{PROJECT_ROOT}/mlruns")
mlflow.set_experiment("patient-readmission-risk")

with mlflow.start_run(run_name="XGBoost_readmission"):
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=len(y_train[y_train == 0]) / max(len(y_train[y_train == 1]), 1),
        eval_metric='logloss',
        random_state=42,
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='roc_auc')

    xgb_model.fit(X_train, y_train)
    y_pred = xgb_model.predict(X_test)
    y_prob = xgb_model.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, y_prob)

    mlflow.log_param('n_features', len(ALL_FEATURES))
    mlflow.log_param('n_estimators', 200)
    mlflow.log_param('dataset', 'MIMIC-III Demo')
    mlflow.log_metric('cv_auc_mean', cv_scores.mean())
    mlflow.log_metric('cv_auc_std', cv_scores.std())
    mlflow.log_metric('test_auc', test_auc)
    mlflow.sklearn.log_model(xgb_model, 'XGBoost_readmission')

print(f"\n{'=' * 45}")
print(f"Cross-validation AUC: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
print(f"Test AUC:             {test_auc:.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=['Not Readmitted', 'Readmitted']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['Not Readmitted', 'Readmitted'],
).plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix - 30-Day Readmission Model')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data' / 'confusion_matrix.png', dpi=150)
plt.show()

print("\n[OK] Baseline model training complete and logged to MLflow.")


In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# -- Leave-one-out cross-validation --------------------
print("[INFO] Running leave-one-out cross-validation...")
print("This evaluation is appropriate for a small cohort with limited positive cases.\n")

loo = LeaveOneOut()

xgb_loo = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    scale_pos_weight=len(y[y == 0]) / max(len(y[y == 1]), 1),
    eval_metric='logloss',
    random_state=42,
)

y_prob_loo = cross_val_predict(
    xgb_loo,
    X_imputed,
    y,
    cv=loo,
    method='predict_proba',
)[:, 1]
y_pred_loo = (y_prob_loo > 0.3).astype(int)

loo_auc = roc_auc_score(y, y_prob_loo)

print(f"{'=' * 45}")
print(f"Leave-one-out AUC: {loo_auc:.3f}")
print("\nClassification report:")
print(classification_report(y, y_pred_loo, target_names=['Not Readmitted', 'Readmitted']))

# -- SHAP-based feature interpretation -----------------
print("[INFO] Computing SHAP values...")

xgb_loo.fit(X_imputed, y)
explainer = shap.TreeExplainer(xgb_loo)
shap_vals = explainer.shap_values(X_imputed)

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_vals,
    X_imputed,
    feature_names=ALL_FEATURES,
    max_display=15,
    show=False,
)
plt.title('SHAP Feature Importance - 30-Day Readmission\n(MIMIC-III ICU Cohort, n=129)')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data' / 'shap_readmission.png', dpi=150)
plt.show()

print("\n[INFO] Clinical interpretation of leading features:")
print("=" * 50)

feature_importance = pd.DataFrame({
    'feature': ALL_FEATURES,
    'importance': abs(shap_vals).mean(axis=0),
}).sort_values('importance', ascending=False).head(10)

clinical_meanings = {
    'creatinine_mean': 'Elevated creatinine is associated with kidney dysfunction and higher readmission risk.',
    'bun_mean': 'High BUN may reflect kidney stress and greater overall illness severity.',
    'los_days': 'Longer ICU stays often indicate greater clinical complexity at discharge.',
    'age': 'Older patients may have less physiologic reserve after discharge.',
    'comorbidity_count': 'A higher comorbidity burden generally increases follow-up risk.',
    'has_renal': 'Renal disease is a common chronic driver of repeat admissions.',
    'has_cardiac': 'Cardiac disease often requires ongoing monitoring after discharge.',
    'hemoglobin_mean': 'Lower hemoglobin may indicate anemia and reduced recovery capacity.',
    'white_blood_cell_mean': 'Elevated white blood cell count can indicate infection or inflammation.',
    'glucose_mean': 'Glucose instability may reflect diabetes-related complications.',
}

for _, row in feature_importance.iterrows():
    meaning = clinical_meanings.get(row['feature'], 'General clinical risk marker.')
    print(f"\n{row['feature']}")
    print(f"  {meaning}")

with mlflow.start_run(run_name='XGBoost_LOO_CV'):
    mlflow.log_metric('loo_auc', loo_auc)
    mlflow.log_param('cv_strategy', 'LeaveOneOut')
    mlflow.log_param('n_patients', len(y))
    mlflow.log_param('readmission_rate', y.mean())
    mlflow.sklearn.log_model(xgb_loo, 'XGBoost_LOO')

print(f"\n[OK] Leave-one-out AUC: {loo_auc:.3f}")
print("[OK] SHAP analysis saved.")
print("[OK] Results logged to MLflow.")


In [ ]:
# -- Limitations and subgroup review -------------------
print("[INFO] Model assessment and dataset limitations")
print("=" * 55)
print(
    """
Dataset limitations:
- n=129 patients in the MIMIC-III demo cohort
- Only 11 readmission events (8.5% base rate)
- Limited sample size for stable predictive modeling
- Full MIMIC-III would provide much stronger coverage for model development

This project is best understood as a portfolio-quality research exercise rather than a production-ready model.
Clinical deployment would require larger-scale validation and stronger governance review.
"""
)

print("[INFO] Fairness and subgroup analysis")
print("=" * 55)

fairness_results = {}

gender_fair = df.groupby('gender')['readmitted_30d'].agg(['mean', 'count', 'sum'])
gender_fair.columns = ['readmission_rate', 'total', 'readmitted']
print("\nBy gender:")
print(gender_fair.round(3))
fairness_results['gender'] = gender_fair

insurance_fair = df.groupby('insurance')['readmitted_30d'].agg(['mean', 'count', 'sum'])
insurance_fair.columns = ['readmission_rate', 'total', 'readmitted']
print("\nBy insurance type:")
print(insurance_fair.round(3))
fairness_results['insurance'] = insurance_fair

age_fair = df.groupby('age_group', observed=True)['readmitted_30d'].agg(['mean', 'count', 'sum'])
age_fair.columns = ['readmission_rate', 'total', 'readmitted']
print("\nBy age group:")
print(age_fair.round(3))
fairness_results['age_group'] = age_fair

def simplify_ethnicity(value):
    text = str(value)
    if 'WHITE' in text:
        return 'WHITE'
    if 'HISPANIC' in text:
        return 'HISPANIC'
    if 'BLACK' in text:
        return 'BLACK'
    return 'OTHER'

df['ethnicity_simple'] = df['ethnicity'].apply(simplify_ethnicity)
ethnicity_fair = df.groupby('ethnicity_simple')['readmitted_30d'].agg(['mean', 'count', 'sum'])
ethnicity_fair.columns = ['readmission_rate', 'total', 'readmitted']
print("\nBy ethnicity:")
print(ethnicity_fair.round(3))
fairness_results['ethnicity'] = ethnicity_fair

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    'Clinical Fairness Analysis - 30-Day Readmission Risk\nMIMIC-III ICU Cohort (n=129)',
    fontsize=14,
    fontweight='bold',
)

gender_fair['readmission_rate'].plot(
    kind='bar',
    ax=axes[0, 0],
    color=['#3498db', '#e74c3c'],
    edgecolor='black',
    title='Readmission Rate by Gender',
)
axes[0, 0].set_ylabel('Readmission Rate')
axes[0, 0].tick_params(axis='x', rotation=0)
for i, value in enumerate(gender_fair['readmission_rate']):
    axes[0, 0].text(i, value + 0.002, f'{value:.1%}', ha='center', fontweight='bold')

insurance_fair['readmission_rate'].plot(
    kind='bar',
    ax=axes[0, 1],
    color=['#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'],
    edgecolor='black',
    title='Readmission Rate by Insurance Type',
)
axes[0, 1].set_ylabel('Readmission Rate')
axes[0, 1].tick_params(axis='x', rotation=15)
for i, value in enumerate(insurance_fair['readmission_rate']):
    axes[0, 1].text(i, value + 0.002, f'{value:.1%}', ha='center', fontweight='bold')

age_fair['readmission_rate'].plot(
    kind='bar',
    ax=axes[1, 0],
    color='#3498db',
    edgecolor='black',
    title='Readmission Rate by Age Group',
)
axes[1, 0].set_ylabel('Readmission Rate')
axes[1, 0].tick_params(axis='x', rotation=0)
for i, value in enumerate(age_fair['readmission_rate']):
    axes[1, 0].text(i, value + 0.002, f'{value:.1%}', ha='center', fontweight='bold')

comorbidity_rates = df.groupby('comorbidity_count')['readmitted_30d'].mean()
comorbidity_rates.plot(
    kind='bar',
    ax=axes[1, 1],
    color='#e74c3c',
    edgecolor='black',
    title='Readmission Rate by Comorbidity Count\n(Charlson Index Proxy)',
)
axes[1, 1].set_xlabel('Number of Comorbidities')
axes[1, 1].set_ylabel('Readmission Rate')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data' / 'fairness_analysis.png', dpi=150)
plt.show()

print("\n[OK] Fairness analysis complete.")
print("[OK] Saved fairness_analysis.png")

print("\nKey findings:")
print("=" * 55)
print(
    """
1. Hemoglobin and length of stay remain among the strongest risk signals in the model.
2. Readmission rates vary across demographic and insurance subgroups in the demo cohort.
3. Comorbidity burden continues to behave as a clinically plausible driver of readmission.
4. The dataset is too small for production claims, so results should be interpreted cautiously.
5. Leave-one-out cross-validation is more appropriate than a conventional split for this cohort size.
"""
)
